# Metadata Upgrade Status Analysis
Analyze upgrade success rates across projects using the metadata_upgrade_status_prod table

In [1]:
import pandas as pd
import altair as alt
from zombie_squirrel import custom
from aind_data_access_api.document_db import MetadataDbClient

In [2]:
df = custom("metadata_upgrade_status_prod")
print(f"RDS table shape: {df.shape}")
print(f"\nRDS table columns: {df.columns.tolist()}")


RDS table shape: (87460, 5)

RDS table columns: ['v1_id', 'v2_id', 'upgrader_version', 'last_modified', 'status']


In [3]:
client = MetadataDbClient(
    host="api.allenneuraldynamics.org",
    version="v1",
)

extra_columns = {
    "_id": 1,
    "data_description.data_level": 1,
    "data_description.project_name": 1,
    "name": 1,
}

print("\nRetrieving project_name and data_level from DocDB...")
all_records = client.retrieve_docdb_records(
    filter_query={},
    projection={"_id": 1},
    limit=0,
)
all_ids = [record["_id"] for record in all_records]
print(f"Found {len(all_ids)} total records in DocDB")

batch_size = 100
records = []
for start_idx in range(0, len(all_ids), batch_size):
    end_idx = start_idx + batch_size
    batch_ids = all_ids[start_idx:end_idx]
    filter_query = {"_id": {"$in": batch_ids}}
    batch_records = client.retrieve_docdb_records(
        filter_query=filter_query,
        projection=extra_columns,
        limit=0,
    )
    records.extend(batch_records)

for i, record in enumerate(records):
    data_description = record.get("data_description", {})
    if data_description:
        record["data_level"] = data_description.get("data_level", None)
        record["project_name"] = data_description.get("project_name", None)
        record.pop("data_description")
    records[i] = record

extra_df = pd.DataFrame(records)
print(f"Retrieved {len(extra_df)} records from DocDB")

print("\nMerging tables...")
df = df.merge(extra_df, how="left", left_on="v1_id", right_on="_id")
print(f"Merged table shape: {df.shape}")
print(f"Merged table columns: {df.columns.tolist()}")


Retrieving project_name and data_level from DocDB...
Found 87623 total records in DocDB
Retrieved 87623 records from DocDB

Merging tables...
Merged table shape: (87460, 9)
Merged table columns: ['v1_id', 'v2_id', 'upgrader_version', 'last_modified', 'status', '_id', 'name', 'data_level', 'project_name']


In [4]:
df.head(10)

,v1_id,v2_id,upgrader_version,last_modified,status,_id,name,data_level,project_name
0,000099c0-eab2-4d83-bcdc-440954c1e60d,a82c2998-6e26-47de-baea-b12c425ba30f,0.13.4,2026-03-10T04:48:31.581Z,success,000099c0-eab2-4d83-bcdc-440954c1e60d,behavior_775510_2025-04-13_15-30-30_processed_...,derived,AIND Viral Genetic Tools
1,0002954e-88c0-4878-8417-241cc07e432c,7893efe0-2557-4f50-8169-11847acb9cad,0.13.4,2026-03-10T04:54:07.384Z,success,0002954e-88c0-4878-8417-241cc07e432c,behavior_792288_2025-07-22_08-38-59_processed_...,derived,Discovery-Neuromodulator circuit dynamics duri...
2,0002fb98-ffcd-419c-b103-6536d4b46da7,9633f912-ebe4-4da5-8487-de727ed3bd26,0.13.4,2026-04-09T07:21:47.684Z,success,0002fb98-ffcd-419c-b103-6536d4b46da7,behavior_754580_2024-11-27_10-00-49_processed_...,derived,Cognitive flexibility in patch foraging
3,00054555-cfdc-490f-8d0e-c350da4156dc,70edd757-3b60-4282-9ef4-dcf73a42ef1e,0.13.4,2024-11-26T20:53:33.359904Z,success,00054555-cfdc-490f-8d0e-c350da4156dc,behavior_638573_2022-11-10_13-55-38,raw,Dynamic Routing
4,0006c8b3-ad7c-4a0f-8782-481651bc0aef,56285f2e-70e2-4192-b1e7-81940667e8e4,0.13.4,2026-04-17T06:45:38.297Z,success,0006c8b3-ad7c-4a0f-8782-481651bc0aef,multiplane-ophys_846289_2026-04-16_13-56-40,raw,OpenScope
5,00075070-719b-4575-9c4c-2d81ec241db3,abb8a78f-c63e-48f2-9602-9567716ba927,0.13.4,2024-11-26T20:26:30.729780Z,success,00075070-719b-4575-9c4c-2d81ec241db3,behavior_676909_2023-11-03_09-16-17,raw,Dynamic Routing
6,0007a18c-e1c3-426d-af54-8d5a1aeab018,48819b80-b35e-4577-99c6-08bbe8163206,0.13.4,2026-03-10T04:55:23.467Z,success,0007a18c-e1c3-426d-af54-8d5a1aeab018,behavior_770803_2025-02-13_13-04-57,raw,Behavior Platform
7,0008d481-1376-4be9-92d0-c61d1785c7ad,None,0.13.4,2025-10-31T20:15:57.710Z,failed,0008d481-1376-4be9-92d0-c61d1785c7ad,HCR_732195-ROI2-cell15_2024-06-15_06-00-00,raw,None
8,00099710-d776-42c1-ac3a-bd45985cc2b6,None,0.13.4,2025-10-31T20:16:43.663Z,failed,00099710-d776-42c1-ac3a-bd45985cc2b6,HCR_772646-5a-4_2025-07-15_10-00-00,raw,None
9,000ab43e-4841-4bef-8e06-b8a99828cff3,48866135-6ac6-427b-9f84-706ddbaf3f51,0.13.4,2026-03-10T04:49:34.788Z,success,000ab43e-4841-4bef-8e06-b8a99828cff3,behavior_786237_2025-07-02_09-03-01,raw,Discovery-Neuromodulator circuit dynamics duri...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87460 entries, 0 to 87459
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   v1_id             87460 non-null  object
 1   v2_id             66683 non-null  object
 2   upgrader_version  87460 non-null  object
 3   last_modified     87460 non-null  object
 4   status            87460 non-null  object
 5   _id               87460 non-null  object
 6   name              87460 non-null  object
 7   data_level        84974 non-null  object
 8   project_name      75909 non-null  object
dtypes: object(9)
memory usage: 6.0+ MB


In [6]:
df.describe(include='all')

,v1_id,v2_id,upgrader_version,last_modified,status,_id,name,data_level,project_name
count,87460,66683,87460,87460,87460,87460,87460,84974,75909
unique,87460,66681,1,81649,2,87460,86870,5,72
top,000099c0-eab2-4d83-bcdc-440954c1e60d,c7d1811c-8090-4bdb-8f44-e886aa33ac09,0.13.4,2025-08-25T21:31:53.116Z,success,000099c0-eab2-4d83-bcdc-440954c1e60d,behavior_746346_2025-01-09_10-23-01_processed_...,raw,Dynamic Routing
freq,1,2,87460,66,66683,1,52,43643,11624


In [7]:
print("Status values:")
print(df['status'].value_counts())
print(f"\nProject names ({df['project_name'].nunique()} unique):") 
print(df['project_name'].value_counts().head(20))

Status values:
status
success    66683
failed     20777
Name: count, dtype: int64

Project names (72 unique):
project_name
Dynamic Routing                                                                                                                             11624
Behavior Platform                                                                                                                           10757
Genetic Perturbation Platform                                                                                                                7578
Discovery-Neuromodulator circuit dynamics during foraging - Subproject 3 Fiber Photometry Recordings of NM Release During Behavior           7277
Cognitive flexibility in patch foraging                                                                                                      7090
Learning mFISH-V1omFISH                                                                                                                      4110
D

In [8]:
success_by_project = df.groupby('project_name').size().reset_index(name='total_records')
success_by_project = success_by_project.sort_values('total_records', ascending=False)
print("Records by project:")
print(success_by_project)

Records by project:
                                         project_name  total_records
28                                    Dynamic Routing          11624
6                                   Behavior Platform          10757
31                      Genetic Perturbation Platform           7578
26  Discovery-Neuromodulator circuit dynamics duri...           7277
14            Cognitive flexibility in patch foraging           7090
..                                                ...            ...
44                               MesoscopeDevelopment              2
18                                             DevCCF              2
15                               Costa Greene Adrenal              2
56                      Single Neuron Reconstructions              1
71                                            unknown              1

[72 rows x 2 columns]


In [11]:
success_pct_by_project = df.groupby('project_name').apply(
    lambda x: (x['status'] == 'success').sum() / len(x) * 100
).reset_index(name='success_percentage')
success_pct_by_project = success_pct_by_project.sort_values('success_percentage', ascending=False)
print("Success percentage by project:")
print(success_pct_by_project)

Success percentage by project:
                                       project_name  success_percentage
36                                             ISIx               100.0
33                                             HIVE               100.0
1   AIND Scientific Activities - Molecular and proj               100.0
37                             Information Foraging               100.0
39                              LearningmFISHTask1A               100.0
..                                              ...                 ...
7                        Behavior and Motor Control                 0.0
68                                     V1 Deep Dive                 0.0
58                                       Standalone                 0.0
45                                Molecular Anatomy                 0.0
71                                          unknown                 0.0

[72 rows x 2 columns]


/var/folders/dj/b6xhnrzd69s_5s9jyt2r8cfr0000gn/T/ipykernel_63391/258911849.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  success_pct_by_project = df.groupby('project_name').apply(


In [12]:
chart_success = alt.Chart(success_pct_by_project).mark_bar(color='steelblue').encode(
    x=alt.X('project_name:N', sort='-y', title='Project Name'),
    y=alt.Y('success_percentage:Q', title='Success Percentage (%)', scale=alt.Scale(domain=[0, 100])),
    tooltip=['project_name:N', alt.Tooltip('success_percentage:Q', format='.2f')]
).properties(
    width=1000,
    height=500,
    title='Upgrade Success Rate by Project'
)
chart_success.display()

alt.Chart(...)

In [10]:
success_detail = df.groupby(['project_name', 'status']).size().reset_index(name='count')
chart_stacked = alt.Chart(success_detail).mark_bar().encode(
    x=alt.X('project_name:N', sort='-y', title='Project Name'),
    y=alt.Y('count:Q', title='Count'),
    color=alt.Color('status:N', title='Status'),
    tooltip=['project_name:N', 'status:N', 'count:Q']
).properties(
    width=1000,
    height=500,
    title='Upgrade Status Breakdown by Project'
).interactive()
chart_stacked.display()

alt.Chart(...)

In [11]:
print(f"Total records: {len(df)}")
print(f"Total projects analyzed: {df['project_name'].nunique()}")
print(f"\nOverall status distribution:")
print(df['status'].value_counts())
print(f"\nOverall success rate: {(df['status'] == 'success').sum() / len(df) * 100:.2f}%")

Total records: 81705
Total projects analyzed: 68

Overall status distribution:
status
success    45482
failed     36223
Name: count, dtype: int64

Overall success rate: 55.67%


In [ ]:
if status_column and len(success_pct_by_project) > 0:
    chart_success = alt.Chart(success_pct_by_project).mark_bar(color='steelblue').encode(
        x=alt.X('project_name:N', sort='-y', title='Project Name'),
        y=alt.Y('success_percentage:Q', title='Success Percentage (%)', scale=alt.Scale(domain=[0, 100])),
        tooltip=['project_name:N', alt.Tooltip('success_percentage:Q', format='.2f')]
    ).properties(
        width=800,
        height=400,
        title=f'Upgrade Success Rate by Project ({status_column})'
    )
    chart_success.display()

In [ ]:
if status_column:
    success_detail = df.groupby(['project_name', status_column]).size().reset_index(name='count')
    chart_stacked = alt.Chart(success_detail).mark_bar().encode(
        x=alt.X('project_name:N', sort='-y', title='Project Name'),
        y=alt.Y('count:Q', title='Count'),
        color=alt.Color(f'{status_column}:N', title=status_column),
        tooltip=['project_name:N', status_column, 'count:Q']
    ).properties(
        width=800,
        height=400,
        title=f'Upgrade Status Breakdown by Project'
    ).interactive()
    chart_stacked.display()

## Summary Statistics

In [ ]:
print(f"Total records: {len(df)}")
print(f"Total projects: {df['project_name'].nunique()}")
if status_column:
    print(f"\nOverall status distribution:")
    print(df[status_column].value_counts())
    print(f"\nMost common status: {df[status_column].mode()[0]}")